# Dynamic Conditional Correlation Modelling 

*By Vlad Popa*

Squared returns of assets act as a proxy for time-varying variance, revealing volatility clustering: the variability of returns in high-volatility periods tend to cluster together, and low-volatility periods follow one another, rather than occurring randomly. This proves that market volatility is not entirely random, and plays a significant role in risk management, derivative pricing and algorithmic trading.

In this project, we assess the time-varying, conditional correlation between several assets, driven by geopolitical choke points in 2026 using the DCC-GARCH(1,1) model. 

## 1. Import Libraries and Load Data

In [33]:
# Import libraries
library(rugarch)
library(purrr)

# Read .csv files
bdi <- read.csv("input/mod/Modified Baltic Dry Index Historical Data.csv")
oil <- read.csv("input/mod/Modified Crude Oil WTI Futures Historical Data.csv")
soxx <- read.csv("input/mod/Modified SOXX ETF Stock Price History.csv")
urth <- read.csv("input/mod/Modified URTH ETF Stock Price History.csv")
gold <- read.csv("input/mod/Modified XAU_USD Historical Data.csv")

In [34]:
# Save returns of each asset to a separate dataframe
bdi_rets <- data.frame(Date = bdi$Date,  BDI = bdi$Return / 100)
oil_rets <- data.frame(Date = oil$Date,  OIL = oil$Return / 100)
soxx_rets <- data.frame(Date = soxx$Date, SOXX = soxx$Return / 100)
urth_rets <- data.frame(Date = urth$Date, URTH = urth$Return / 100)
gold_rets <- data.frame(Date = gold$Date, GOLD = gold$Return / 100)

# Save all dataframes to a list
rets_list <- list(bdi_rets, oil_rets, soxx_rets, urth_rets, gold_rets)

# Sequentially merge each dataframe on the 'Date' column
returns_scaled <- purrr::reduce(rets_list, merge, by = "Date")
rownames(returns_scaled) <- returns_scaled$Date
returns_scaled$Date <- NULL

## 2. GARCH Modelling

In [39]:
# Function for modelling conditional volatilities using eGARCH
my_garch <- function(returns_scaled){
    # Specify our GARCH model: how do individual variances behave?
    uspec <- ugarchspec(
        mean.model = list(armaOrder = c(0,0), include.mean = FALSE),
        variance.model = list(model = "eGARCH"),
        distribution.model = "norm")

    # Replicate the univariate GARCH model across all assets
    mspec <- multispec(replicate(ncol(returns_scaled), uspec))       

    # Estimate all conditional volatilities using our defined GARCH model
    fitlist <- multifit(multispec = mspec, data = returns_scaled)
    sigmas <- sigma(fitlist)
    rownames(sigmas) <- rownames(returns_scaled)
    colnames(sigmas) <- colnames(returns_scaled)
    
    # Save specifications, fits and conditional volatilities together
    output = list(
        sigmas = sigmas,
        mspec = mspec,
        fitlist = fitlist
    )
    return (output)
}

# Run function and save conditional volatilites
ugarch_results <- my_garch(returns_scaled) 
head(ugarch_results[["sigmas"]])

,BDI,OIL,SOXX,URTH,GOLD
2026-01-02,0.02765502,0.04374308,0.02827338,0.009258129,0.02091431
2026-01-05,0.02455597,0.04047204,0.02900426,0.009076927,0.01856148
2026-01-06,0.02447209,0.03979490,0.02866007,0.008613400,0.02271416
2026-01-07,0.02352242,0.03663624,0.02902499,0.008374815,0.02169813
2026-01-08,0.02610684,0.03381557,0.02871770,0.008773055,0.01986332
2026-01-09,0.02853824,0.03718718,0.02872888,0.008909618,0.01835290


## 3. DCC Modelling